## 🌳 LeetCode 104: Maximum Depth of Binary Tree
---

<div style="padding: 15px; border-left: 8px solid #1a8a5a; background-color: #e8f5e9; color: #1b5e20; border-radius: 4px;">
    <strong>The Core Insight:</strong> Depth is recursive by nature. The depth of a tree is 1 (the root)
    plus the depth of its deeper subtree. The base case: an empty node has depth 0.
    DFS recursive solves this in 3 lines.
</div>

### 📋 Problem

Given the root of a binary tree, return its **maximum depth** — the number of nodes along
the longest path from the root down to the farthest leaf node.

```
    3          maxDepth = 3
   / \
  9  20        (path: 3 → 20 → 15 or 3 → 20 → 7)
    /  \
   15   7
```

**Constraints:** 0 ≤ nodes ≤ 10⁴ | -100 ≤ Node.val ≤ 100


### 🧠 Choose Your Mental Model

| Model | The "Story" | Mechanism |
|-------|-------------|-----------|
| **Russian Dolls** | "I'm a doll. My depth is 1 + the depth of my bigger half." Each call returns depth for its subtree, plus 1. | Recursive: `1 + max(left, right)` |
| **DFS Floor Counter** | "I'm exploring a cave. I count steps as I go down. At every dead-end (leaf), I report how deep I am." | Count levels on the way back up |
| **BFS Level Counter** | "I process the tree floor by floor. Each time I clear a floor, I increment the floor counter." | Queue + level snapshot |


In [ ]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(vals):
    """Build tree from level-order list (None = missing node)."""
    if not vals or vals[0] is None:
        return None
    from collections import deque
    root = TreeNode(vals[0])
    q = deque([root])
    i = 1
    while q and i < len(vals):
        node = q.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i])
            q.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i])
            q.append(node.right)
        i += 1
    return root

print("TreeNode ready.")

def maxDepth_brute(root: TreeNode) -> int:
    """
    Brute force: collect ALL root-to-leaf paths, return length of longest.
    Time: O(n)  |  Space: O(n * h) — stores every path
    Pedagogically clear but wastes memory: each leaf stores full path.
    """
    if not root:
        return 0

    paths = []

    def dfs(node, path):
        if not node:
            return
        path.append(node.val)
        if not node.left and not node.right:   # leaf
            paths.append(list(path))            # copy path (don't store reference)
        dfs(node.left,  path)
        dfs(node.right, path)
        path.pop()                              # backtrack

    dfs(root, [])
    return max(len(p) for p in paths) if paths else 0


# Test
root = build_tree([3, 9, 20, None, None, 15, 7])
print("Brute force:", maxDepth_brute(root))   # Expected: 3

root2 = build_tree([1, None, 2])
print("Brute force:", maxDepth_brute(root2))  # Expected: 2


## 🔬 Complexity Analysis: Brute Force vs Optimal

<div style="padding: 15px; border-left: 8px solid #1a8a5a; background-color: #f1f8f4; color: #1b5e20; border-radius: 4px;">
    <strong>The Core Insight:</strong> Both approaches are O(n) time — we must visit every node.
    The difference is space: brute force stores all paths (O(n × h)), while optimal DFS
    only stores the call stack (O(h)), and BFS stores at most one level (O(w)).
</div>

---

## 📉 Why Brute Force Uses More Space

Collecting all root-to-leaf paths:
- A balanced tree of depth h has ~n/2 leaves
- Each path has length h
- Total space: O(n × h) = O(n log n) for balanced, O(n²) for skewed

| Approach | Time | Space | Notes |
|----------|------|-------|-------|
| Brute force (collect paths) | O(n) | O(n × h) | Stores every path |
| **DFS Recursive (optimal)** | **O(n)** | **O(h)** | Call stack only |
| BFS Iterative | O(n) | O(w) | w = max level width |

### Skewed tree space comparison
- Brute force: O(n²) — each of n leaves has a path of length n
- DFS: O(n) — call stack grows to depth n
- BFS: O(1) — only 1 node per level → queue never exceeds size 1

<div style="padding: 10px; border-left: 8px solid #4CAF50; background-color: #f1f8f4; color: #1b5e20; border-radius: 4px;">
    <strong>✅ Summary:</strong> Use DFS recursive for clean code (O(h) space).
    Prefer BFS iterative when stack overflow is a concern (very deep unbalanced trees).
</div>


In [ ]:
from collections import deque

def maxDepth(root: TreeNode) -> int:
    """
    Optimal DFS Recursive — 3 lines.
    Time: O(n)  |  Space: O(h) — recursion call stack
    h = O(log n) balanced, O(n) worst case (skewed tree)
    """
    if not root:
        return 0
    return 1 + max(maxDepth(root.left), maxDepth(root.right))


def maxDepth_bfs(root: TreeNode) -> int:
    """
    BFS Iterative — counts levels explicitly.
    Time: O(n)  |  Space: O(w) — w = max width of tree
    Preferred when tree may be highly skewed (avoids deep recursion).
    """
    if not root:
        return 0
    queue = deque([root])
    depth = 0

    while queue:
        depth += 1
        for _ in range(len(queue)):      # process entire current level
            node = queue.popleft()
            if node.left:  queue.append(node.left)
            if node.right: queue.append(node.right)

    return depth


# Tests
root = build_tree([3, 9, 20, None, None, 15, 7])
print("DFS:", maxDepth(root))       # 3
print("BFS:", maxDepth_bfs(root))   # 3

root2 = build_tree([1, None, 2])
print("DFS:", maxDepth(root2))      # 2
print("BFS:", maxDepth_bfs(root2))  # 2

root3 = None
print("DFS empty:", maxDepth(root3))   # 0


## 🎤 5 Interview Q&A

### Q1: What is the space complexity of the recursive DFS solution?

**Answer:** O(h) where h is the tree height. In the worst case (a completely skewed tree —
every node has only one child), h = n, giving O(n) space. For a balanced binary tree,
h = log n, giving O(log n) space. The space is the call stack — one frame per level of recursion.

---

### Q2: When is BFS preferred over DFS for finding max depth?

**Answer:** When the tree might be highly skewed and you're worried about stack overflow from
deep recursion. Python's default recursion limit is ~1,000. A skewed tree with 10,000 nodes
would cause a RecursionError with DFS. BFS uses an explicit queue and has no recursion limit.
In practice, for balanced trees, DFS is more concise and equally performant.

---

### Q3: Why does `1 + max(left, right)` work? Walk me through the recursion.

**Answer:** For a leaf node: `left = maxDepth(None) = 0`, `right = maxDepth(None) = 0`.
So leaf depth = `1 + max(0, 0) = 1`. ✓
For an internal node: it returns `1 (for itself) + max(depth of left subtree, depth of right subtree)`.
This correctly propagates depth bottom-up. The recursion bottoms out at None → 0.

---

### Q4: What's the difference between height and depth?

**Answer:** Often used interchangeably, but technically:
- **Depth of a node** = distance from root to that node (root has depth 0)
- **Height of a node** = length of longest path from that node to a leaf
- **Height of tree** = height of root = maximum depth of any leaf

LeetCode #104 asks for the height of the root, which equals the maximum depth of any leaf.
The recursive formula `1 + max(left, right)` computes this correctly.

---

### Q5: How would you find the minimum depth of a binary tree?

**Answer:** NOT `1 + min(left, right)` — that's wrong for nodes with only one child
(it would return the depth of the None child = 0, making everything depth 1).
Correct approach: if a node has no left child, recurse only into the right. If no right,
only into left. If both exist, take the min. Or use BFS — the first leaf you encounter
is at minimum depth, so return as soon as you process a leaf.

```python
def minDepth(root):
    if not root: return 0
    if not root.left: return 1 + minDepth(root.right)
    if not root.right: return 1 + minDepth(root.left)
    return 1 + min(minDepth(root.left), minDepth(root.right))
```


## 💼 The Citi Narrative

**Context:** Citi's APM infrastructure had a microservices call chain for trade processing.
Each service depended on downstream services — forming a dependency tree rooted at the
trade submission API. Incidents often cascaded down deep call chains.

**The Problem:** "What is the deepest dependency chain in our critical trading path?"
Chains longer than 6 hops were flagged for re-architecture — deep chains increase latency
and make failure isolation harder.

**How Max Depth Applied:** The service dependency graph was a tree (after cycle detection).
The `maxDepth` algorithm ran nightly on the service registry to compute max call chain depth.
Any chain exceeding 6 was reported to the architecture team.

**Practical extension:** Instead of depth, compute the **critical path latency** —
each node had a `p99_latency_ms` attribute. The "depth" became the maximum sum of latencies
along any root-to-leaf path. This is a simple extension: `max_latency(node) = node.p99 + max(left, right)`.

**Interview Line:** *"At Citi, we modeled the microservices call graph as a tree. The max depth
algorithm ran nightly — chains deeper than 6 services were escalated. We later extended it to
sum p99 latencies instead of counting hops, which is just replacing 1 with node.latency in the
recursion. Same algorithm, richer signal."*


In [ ]:
# ── PRACTICE: Diameter of Binary Tree ──────────────────────────────────────
# Given a binary tree, return the length of the DIAMETER —
# the longest path between ANY two nodes (path may not pass through root).
# The diameter at any node = left_depth + right_depth.
# Hint: track max diameter as a side effect during the depth recursion.
#
# Example: [1,2,3,4,5] → diameter = 3 (path: 4-2-1-3 or 5-2-1-3)

def diameterOfBinaryTree(root: TreeNode) -> int:
    # Your solution here
    pass

# Test
root = build_tree([1, 2, 3, 4, 5])
print(diameterOfBinaryTree(root))    # Expected: 3

# Solution:
def diameterOfBinaryTree_sol(root):
    max_diam = [0]
    def depth(node):
        if not node: return 0
        L = depth(node.left)
        R = depth(node.right)
        max_diam[0] = max(max_diam[0], L + R)   # diameter through this node
        return 1 + max(L, R)
    depth(root)
    return max_diam[0]

print("Diameter:", diameterOfBinaryTree_sol(build_tree([1, 2, 3, 4, 5])))   # 3
print("Diameter:", diameterOfBinaryTree_sol(build_tree([1, 2])))             # 1


## 🎯 Summary: Key Takeaways

### The Pattern
**DFS Recursive:** `maxDepth(root) = 0 if not root else 1 + max(left, right)`

### When to Use Each Approach
- ✅ **DFS Recursive** — clean, 3 lines, default choice for balanced trees
- ✅ **BFS Iterative** — preferred for skewed trees to avoid recursion depth limit
- ❌ **Collect all paths** — O(n × h) space — never in production

### Complexity
| Approach | Time | Space |
|----------|------|-------|
| DFS Recursive | O(n) | O(h) |
| BFS Iterative | O(n) | O(w) |

### Extensions (know these for follow-ups)
- **Minimum depth:** handle single-child nodes explicitly
- **Diameter:** track `left_depth + right_depth` as a side effect
- **Path with max sum:** replace 1 with node.val, handle negatives carefully

### Interview Confidence Checklist
- [ ] Can write DFS solution from memory in 60 seconds
- [ ] Can explain O(h) space complexity with skewed vs balanced example
- [ ] Can write BFS iterative version with level snapshot
- [ ] Can connect to Citi: service dependency chain depth
- [ ] Can extend to diameter (most common follow-up)

**"Simplicity and clarity is Gold."** — Sean's Study Mantra 🚀
